# Beyond price and direction

A common misconception about tsauditor is that it only handles a `price`
column and a `Direction` target. It doesn't — it is **column-agnostic**. This
notebook audits three very different column types on the *real* OGDC dataset:

* **Volume** — must be non-negative
* **RSI** — a bounded oscillator, must sit in [0, 100]
* **OHLC bars** — must satisfy `Low <= Open <= High` and `Low <= High`

None of these are the target, and none involve leakage — they are *domain
validity* rules (VAL001 bounds, VAL002 relations) that you declare and
tsauditor verifies.

In [1]:
import pandas as pd
import tsauditor as tsa

df = pd.read_csv(
    "../ogdc_leakage_case/ogdc_with_regimes.csv", index_col="Date", parse_dates=True
)
df = df.dropna(subset=["Direction"])
print("rows:", len(df))
print("RSI range:", round(df.RSI.min(), 1), "-", round(df.RSI.max(), 1))
print("Volume min:", df.Volume.min())
df[["Open", "High", "Low", "Volume", "RSI"]].head()

rows: 1537
RSI range: 20.2 - 85.7
Volume min: 289100.0


,Open,High,Low,Volume,RSI
Date,,,,,
2020-01-22,148.51,148.51,140.72,19240000.0,62.227181
2020-01-23,145.51,146.00,142.50,3540000.0,60.144533
2020-01-24,143.06,144.75,141.05,2110000.0,61.242135
2020-01-27,142.20,143.90,140.20,1610000.0,54.267752
2020-01-28,141.50,141.50,138.75,1200000.0,52.511358


## The rules, on clean real data

We declare bounds for Volume and RSI, and ordering relations for the OHLC
bars. Real, well-formed market data should satisfy all of them — so a specific
auditor reports **nothing**. (An auditor that fired here would be crying wolf.)

In [2]:
constraints = {
    "bounds": {
        "RSI": {"min": 0, "max": 100},
        "Volume": {"min": 0},
    },
    "relations": [("Low", "High"), ("Low", "Open"), ("Open", "High")],
}

report = tsa.scan(df, constraints=constraints, run_stationarity=False)
validity = report.filter(module="validity")
print("validity issues on clean data:", len(validity))

validity issues on clean data: 0


## Now inject three realistic feed glitches

Exactly the kinds of corruption a live market feed produces — and each one is
invisible to a price/direction-only mindset. We inject them into a **copy** and
clearly label them, then run the same audit.

In [3]:
bad = df.copy()
i = bad.index
bad.loc[i[100], "Volume"] = -500_000.0  # impossible negative volume
bad.loc[i[200], "RSI"] = 130.0  # RSI outside [0, 100]
bad.loc[i[300], "High"] = bad.loc[i[300], "Low"] - 1.0  # crossed bar (High < Low)

report = tsa.scan(bad, constraints=constraints, run_stationarity=False)
for iss in report.filter(module="validity"):
    ev = iss.evidence
    if iss.code == "VAL002":
        what = ev["low_col"] + " <= " + ev["high_col"] + " violated"
    else:
        what = iss.column + " out of bounds"
    print(iss.severity.upper(), iss.code, "|", what, "| n =", ev["n_violations"])

CRITICAL VAL002 | Low <= High violated | n = 1
CRITICAL VAL002 | Open <= High violated | n = 1
WARNING VAL001 | RSI out of bounds | n = 1
WARNING VAL001 | Volume out of bounds | n = 1


Three columns, three column *types*, all caught — and `Direction`/`price`
leakage never entered the picture. VAL002 (the crossed bar) is CRITICAL; the
out-of-range Volume and RSI are VAL001 warnings.

For the other alternative-data column types — **macro** series with a publish
lag (LEK004 as-of) and **sentiment** bounds — see
[`../new_features_walkthrough.ipynb`](../new_features_walkthrough.ipynb).

**Try it on your own data.** Any numeric time-series frame works — e.g. load a
public CSV straight from GitHub and declare the rules your columns must obey:

```python
url = "https://raw.githubusercontent.com/<user>/<repo>/main/data.csv"
df = pd.read_csv(url, index_col=0, parse_dates=True)
tsa.scan(df, constraints={"bounds": {"my_ratio": {"min": 0, "max": 1}}})
```